# 장기 실종아동 나이 변환(Age Progression) 시스템 - Google Colab T4 GPU 원클릭 학습 파이프라인

이 주피터 노트북은 Google Colab의 무료 GPU(T4 16GB) 환경을 활용하여, 친부모 유전 정보(Kinship Cross-Attention)와 생체 안면 정체성 보존(ArcFace)을 결합한 **나이 변환 딥러닝 모델의 실제 데이터 대규모 학습 및 체크포인트 추출**을 수행할 수 있도록 구성되었습니다.

---
### 📌 런타임 유형 확인
상단 메뉴에서 **[런타임] -> [런타임 유형 변경] -> 하드웨어 가속기 [T4 GPU]** 로 설정되어 있는지 확인하세요.

In [ ]:
# 1. GPU 가속기 상태 점검
!nvidia-smi

In [ ]:
# 2. GitHub 저장소 클론 및 작업 디렉토리 이동
!git clone https://github.com/ggg2214/kinship-age-progression.git
%cd kinship-age-progression
!ls -la

In [ ]:
# 3. 필수 의존성 패키지 설치
!pip install -q torch torchvision fastapi uvicorn python-multipart Pillow

In [ ]:
# 4. 실제 얼굴 및 가족 연령 데이터셋 수집 & Triplet 매핑
!python age_progression/data/download_datasets.py --data-dir data_utkface --manifest data_utkface/manifest.json

In [ ]:
# 5. GPU 기반 Full Training 실행 (고해상도 256x256, 10 Epochs)
!python age_progression/train.py \
    --device cuda \
    --image_size 256 \
    --batch_size 4 \
    --lr 0.0002 \
    --epochs 10

In [ ]:
# 6. 학습된 체크포인트로 연령대별 성장 파노라마(10~60세) 시각화 배너 생성
!python age_progression/visualize_samples.py \
    --child sample_data/family_001/child_age06.jpg \
    --father sample_data/family_001/father_age38.jpg \
    --mother sample_data/family_001/mother_age35.jpg \
    --ages 10 20 30 40 50 60 \
    --source-age 6 \
    --checkpoint checkpoints/age_progression_latest.pt \
    --output outputs/colab_timelapse.png

from IPython.display import Image, display
display(Image('outputs/colab_timelapse.png'))

In [ ]:
# 7. 정량 평가 및 소거 연구(Ablation Study) 벤치마크 실행
!python age_progression/evaluate.py \
    --checkpoint checkpoints/age_progression_latest.pt \
    --num-samples 16

In [ ]:
# 8. (선택 사항) 학습된 최종 체크포인트를 Google Drive로 백업하기
# from google.colab import drive
# drive.mount('/content/drive')
# !cp checkpoints/age_progression_latest.pt /content/drive/MyDrive/age_progression_best.pt
# print('[OK] Model checkpoint saved to Google Drive!')